In [79]:
def print_board(board):
    for i in range(9):
        if i%3==0 and i!=0:
            print("-"*12)
        for j in range(9):
            if j%3==0 and j!=0:
                print("|",end="")
            print(board[i][j],end="")
        print()

In [80]:
board = [
    [5,3,0,0,7,0,0,0,0],
    [6,0,0,1,9,5,0,0,0],
    [0,9,8,0,0,0,0,6,0],
    [8,0,0,0,6,0,0,0,3],
    [4,0,0,8,0,3,0,0,1],
    [7,0,0,0,2,0,0,0,6],
    [0,6,0,0,0,0,2,8,0],
    [0,0,0,4,1,9,0,0,5],
    [0,0,0,0,8,0,0,7,9]
]

print_board(board)

530|070|000
600|195|000
098|000|060
------------
800|060|003
400|803|001
700|020|006
------------
060|000|280
000|419|005
000|080|079


In [81]:
def is_valid(board, row, col, num):
    #check if the number was repeated in the row
    if num in board[row]:
        return False

    #check if the number was repeated in the column
    for i in range(9):
        if board[i][col] == num:
            return False

    #check if the number was repeated in the 3x3 box
    start_r, start_c = (row//3)*3, (col//3)*3
    for i in range(3):
        for j in range(3):
            if board[start_r+i][start_c+j] == num:
                return False

    return True

In [82]:
import random
def find_empty(board):
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                return r, c
    return None

def solve_random(board):
    empty = find_empty(board)
    if not empty:
        return True

    r, c = empty

    nums = list(range(1, 10))
    random.shuffle(nums)

    for num in nums:
        if is_valid(board, r, c, num):
            board[r][c] = num

            if solve_random(board):
                return True

            board[r][c] = 0

    return False

In [83]:
import copy
import time

b = copy.deepcopy(board)
start = time.time()
solve_random(b)
end = time.time()

print_board(b)
print("Backtracking time:", end - start)

534|678|912
672|195|348
198|342|567
------------
859|761|423
426|853|791
713|924|856
------------
961|537|284
287|419|635
345|286|179
Backtracking time: 0.013470649719238281


In [84]:
def init_domains(board):
    domains = {}

    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                domains[(r, c)] = set(range(1, 10))
            else:
                domains[(r, c)] = {board[r][c]}

    return domains

In [85]:
def get_neighbors(cell):
    r, c = cell
    neighbors = set()

    for i in range(9):
        neighbors.add((r, i))
        neighbors.add((i, c))

    start_r, start_c = (r//3)*3, (c//3)*3
    for i in range(3):
        for j in range(3):
            neighbors.add((start_r+i, start_c+j))

    neighbors.remove(cell)
    return neighbors

In [86]:
def revise(domains, xi, xj):
    revised = False

    for x in set(domains[xi]):
        if not any(x != y for y in domains[xj]):
            domains[xi].remove(x)
            revised = True

    return revised

In [87]:
from collections import deque
ac3_log=[]
def ac3(domains):
    queue = deque()

    for xi in domains:
        for xj in get_neighbors(xi):
            queue.append((xi, xj))

    step = 0
    ac3_log.clear()  # reset log each run

    while queue:
        xi, xj = queue.popleft()

        snapshot_before = domains[xi].copy()

        changed = revise(domains, xi, xj)

        snapshot_after = domains[xi].copy()

        ac3_log.append({
            "step": step,
            "arc": (xi, xj),
            "before": snapshot_before,
            "after": snapshot_after,
            "changed": changed
        })

        if changed:
            if len(domains[xi]) == 0:
                return False, domains

            for xk in get_neighbors(xi):
                queue.append((xk, xi))

        step += 1

    return True, domains

In [88]:
def print_ac3_tree():
    for item in ac3_log:
        print(f"\nStep {item['step']}")
        print(f"Arc: {item['arc']}")
        print(f"Before: {item['before']}")
        print(f"After: {item['after']}")
        print(f"Changed: {item['changed']}")

In [89]:
def update_board(board, domains):
    changed = False
    #loop over all cells and their corresponding domains
    for (r, c), values in domains.items():
        #if the cell has one value in its domain and this cell is not assigned, then I am sure it has this value
        if len(values) == 1:
            val = list(values)[0]
            if board[r][c] == 0:
                board[r][c] = val
                changed = True

    return changed

In [90]:
def find_unassigned(domains):
    for cell in domains:
        if len(domains[cell]) > 1:
            return cell
    return None


def backtracking_with_domains(domains, board):
    cell = find_unassigned(domains)

    if not cell:
        return True  # solved

    for value in list(domains[cell]):
        if is_valid(board, cell[0], cell[1], value):

            board[cell[0]][cell[1]] = value

            # copy domains 
            new_domains = copy.deepcopy(domains)

            new_domains[cell] = {value}

            if backtracking_with_domains(new_domains, board):
                return True

            board[cell[0]][cell[1]] = 0

    return False

In [91]:
def solve_with_ac3(board):
    domains = init_domains(board)

    while True:
        success, domains = ac3(domains)
        if not success:
            return False

        # keep updating until no changes
        if not update_board(board, domains):
            break

    return backtracking_with_domains(domains, board)

In [92]:
b = copy.deepcopy(board)

start = time.time()
solve_with_ac3(b)
end = time.time()

print_board(b)
print_ac3_tree()
print("AC3 + Backtracking time:", end - start)

534|678|912
672|195|348
198|342|567
------------
859|761|423
426|853|791
713|924|856
------------
961|537|284
287|419|635
345|286|179

Step 0
Arc: ((0, 0), (4, 0))
Before: {5}
After: {5}
Changed: False

Step 1
Arc: ((0, 0), (8, 0))
Before: {5}
After: {5}
Changed: False

Step 2
Arc: ((0, 0), (0, 2))
Before: {5}
After: {5}
Changed: False

Step 3
Arc: ((0, 0), (0, 5))
Before: {5}
After: {5}
Changed: False

Step 4
Arc: ((0, 0), (2, 2))
Before: {5}
After: {5}
Changed: False

Step 5
Arc: ((0, 0), (1, 0))
Before: {5}
After: {5}
Changed: False

Step 6
Arc: ((0, 0), (0, 8))
Before: {5}
After: {5}
Changed: False

Step 7
Arc: ((0, 0), (3, 0))
Before: {5}
After: {5}
Changed: False

Step 8
Arc: ((0, 0), (5, 0))
Before: {5}
After: {5}
Changed: False

Step 9
Arc: ((0, 0), (0, 1))
Before: {5}
After: {5}
Changed: False

Step 10
Arc: ((0, 0), (0, 7))
Before: {5}
After: {5}
Changed: False

Step 11
Arc: ((0, 0), (1, 2))
Before: {5}
After: {5}
Changed: False

Step 12
Arc: ((0, 0), (0, 4))
Before: {5}
After

In [93]:
def generate_full_board():
    board = [[0]*9 for _ in range(9)]
    solve_random(board)
    return board

In [94]:
def generate_puzzle(remove=40):
    board = generate_full_board()
    full = copy.deepcopy(board)

    cells = [(r, c) for r in range(9) for c in range(9)]
    random.shuffle(cells)

    count = 0

    for r, c in cells:
        if count >= remove:
            break

        backup = board[r][c]
        board[r][c] = 0

        count += 1

    return board, full

In [95]:
def run_experiment(board, name):
    print(f"\n--- {name} ---")

    b1 = copy.deepcopy(board)
    start = time.time()
    solve_random(b1)
    t1 = time.time() - start

    b2 = copy.deepcopy(board)
    start = time.time()
    solve_with_ac3(b2)
    t2 = time.time() - start
    print_ac3_tree()
    print("Backtracking time:", t1)
    print("AC3 + Backtracking time:", t2)

In [96]:
easy= generate_puzzle(30)[0]

run_experiment(easy, "Easy")


--- Easy ---

Step 0
Arc: ((0, 0), (4, 0))
Before: {9}
After: {9}
Changed: False

Step 1
Arc: ((0, 0), (8, 0))
Before: {9}
After: {9}
Changed: False

Step 2
Arc: ((0, 0), (0, 2))
Before: {9}
After: {9}
Changed: False

Step 3
Arc: ((0, 0), (0, 5))
Before: {9}
After: {9}
Changed: False

Step 4
Arc: ((0, 0), (2, 2))
Before: {9}
After: {9}
Changed: False

Step 5
Arc: ((0, 0), (1, 0))
Before: {9}
After: {9}
Changed: False

Step 6
Arc: ((0, 0), (0, 8))
Before: {9}
After: {9}
Changed: False

Step 7
Arc: ((0, 0), (3, 0))
Before: {9}
After: {9}
Changed: False

Step 8
Arc: ((0, 0), (5, 0))
Before: {9}
After: {9}
Changed: False

Step 9
Arc: ((0, 0), (0, 1))
Before: {9}
After: {9}
Changed: False

Step 10
Arc: ((0, 0), (0, 7))
Before: {9}
After: {9}
Changed: False

Step 11
Arc: ((0, 0), (1, 2))
Before: {9}
After: {9}
Changed: False

Step 12
Arc: ((0, 0), (0, 4))
Before: {9}
After: {9}
Changed: False

Step 13
Arc: ((0, 0), (2, 1))
Before: {9}
After: {9}
Changed: False

Step 14
Arc: ((0, 0), (7, 0))

In [97]:
medium= generate_puzzle(45)[0]

run_experiment(medium, "Medium")


--- Medium ---

Step 0
Arc: ((0, 0), (4, 0))
Before: {3}
After: {3}
Changed: False

Step 1
Arc: ((0, 0), (8, 0))
Before: {3}
After: {3}
Changed: False

Step 2
Arc: ((0, 0), (0, 2))
Before: {3}
After: {3}
Changed: False

Step 3
Arc: ((0, 0), (0, 5))
Before: {3}
After: {3}
Changed: False

Step 4
Arc: ((0, 0), (2, 2))
Before: {3}
After: {3}
Changed: False

Step 5
Arc: ((0, 0), (1, 0))
Before: {3}
After: {3}
Changed: False

Step 6
Arc: ((0, 0), (0, 8))
Before: {3}
After: {3}
Changed: False

Step 7
Arc: ((0, 0), (3, 0))
Before: {3}
After: {3}
Changed: False

Step 8
Arc: ((0, 0), (5, 0))
Before: {3}
After: {3}
Changed: False

Step 9
Arc: ((0, 0), (0, 1))
Before: {3}
After: {3}
Changed: False

Step 10
Arc: ((0, 0), (0, 7))
Before: {3}
After: {3}
Changed: False

Step 11
Arc: ((0, 0), (1, 2))
Before: {3}
After: {3}
Changed: False

Step 12
Arc: ((0, 0), (0, 4))
Before: {3}
After: {3}
Changed: False

Step 13
Arc: ((0, 0), (2, 1))
Before: {3}
After: {3}
Changed: False

Step 14
Arc: ((0, 0), (7, 0

In [98]:
hard= generate_puzzle(55)[0]

run_experiment(hard, "Hard")


--- Hard ---

Step 0
Arc: ((0, 0), (4, 0))
Before: {1, 2, 3, 4, 5, 6, 7, 8, 9}
After: {1, 2, 3, 4, 5, 6, 7, 8, 9}
Changed: False

Step 1
Arc: ((0, 0), (8, 0))
Before: {1, 2, 3, 4, 5, 6, 7, 8, 9}
After: {1, 2, 3, 4, 5, 7, 8, 9}
Changed: True

Step 2
Arc: ((0, 0), (0, 2))
Before: {1, 2, 3, 4, 5, 7, 8, 9}
After: {1, 2, 3, 4, 5, 7, 8, 9}
Changed: False

Step 3
Arc: ((0, 0), (0, 5))
Before: {1, 2, 3, 4, 5, 7, 8, 9}
After: {1, 2, 3, 4, 5, 7, 8, 9}
Changed: False

Step 4
Arc: ((0, 0), (2, 2))
Before: {1, 2, 3, 4, 5, 7, 8, 9}
After: {1, 2, 3, 4, 5, 7, 9}
Changed: True

Step 5
Arc: ((0, 0), (1, 0))
Before: {1, 2, 3, 4, 5, 7, 9}
After: {1, 2, 3, 4, 5, 7, 9}
Changed: False

Step 6
Arc: ((0, 0), (0, 8))
Before: {1, 2, 3, 4, 5, 7, 9}
After: {1, 2, 3, 4, 5, 7, 9}
Changed: False

Step 7
Arc: ((0, 0), (3, 0))
Before: {1, 2, 3, 4, 5, 7, 9}
After: {1, 2, 3, 4, 5, 7, 9}
Changed: False

Step 8
Arc: ((0, 0), (5, 0))
Before: {1, 2, 3, 4, 5, 7, 9}
After: {1, 2, 3, 4, 5, 7, 9}
Changed: False

Step 9
Arc: ((0

In [99]:
def ai_mode():
    board,_ = generate_puzzle(60)

    print("Generated Puzzle:")
    print_board(board)

    solve_with_ac3(board)

    print("\nSolved Puzzle:")
    print_board(board)

    print("\nAC3 Trace:")
    print_ac3_tree()

In [100]:
def is_board_valid(board):
    # check all filled cells
    for r in range(9):
        for c in range(9):
            if board[r][c] != 0:
                num = board[r][c]

                # temporarily remove it
                board[r][c] = 0

                if not is_valid(board, r, c, num):
                    board[r][c] = num
                    return False

                board[r][c] = num

    return True

In [101]:
def user_mode():
    print("Enter Sudoku (use 0 for empty cells):")

    board = []

    for i in range(9):
        row = list(map(int, input(f"Row {i+1}: ").split()))
        board.append(row)
    if not is_board_valid(board):
        print("Invalid puzzle! It violates Sudoku rules.")
        return
    print("\nYour Puzzle:")
    print_board(board)

    if solve_with_ac3(board):
        print("\nSolved Puzzle:")
        print_board(board)
    else:
        print("No solution exists!")

In [105]:
print("1 → AI solves generated puzzle")
print("2 → User enters puzzle")

choice = input("Choose mode: ")

if choice == "1":
    ai_mode()
elif choice == "2":
    user_mode()
else:
    print("Invalid choice")

1 → AI solves generated puzzle
2 → User enters puzzle
Enter Sudoku (use 0 for empty cells):

Your Puzzle:
000|260|701
680|070|090
190|004|500
------------
820|100|040
004|602|900
050|003|028
------------
009|300|074
040|050|036
703|018|000

Solved Puzzle:
435|269|781
682|571|493
197|834|562
------------
826|195|347
374|682|915
951|743|628
------------
519|326|874
248|957|136
763|418|259
